In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load v5 dataset

data_path = "data/compiled_model_ready/MR_cities_worldpop_knn_macro_v5.csv"
df = pd.read_csv(data_path)

print("Loaded v5 shape:", df.shape)
print(df.head())

Loaded v5 shape: (509, 77)
           city           country    country_norm_x        lat         lon  \
0       Caracas         Venezuela         venezuela  10.506093  -66.914601   
1      Pretoria      South Africa      south africa -25.745928   28.187910   
2        Durban      South Africa      south africa -29.861825   31.009909   
3  Port Moresby  Papua New Guinea  papua new guinea  -9.474330  147.159950   
4  Johannesburg      South Africa      south africa -26.205000   28.049722   

   crime_index  safety_index  crimeindex_2023  crimeindex_2020  \
0         83.6          16.4            83.76            84.49   
1         82.0          18.0            76.86            77.49   
2         81.0          19.0            76.86            77.49   
3         80.7          19.3            80.79            81.93   
4         80.7          19.3            76.86            77.49   

   safetyindex_2020  ...                     country_norm_y       gdp  \
0             15.51  ...  venezuel

In [2]:
# Define v4 feature groups

TARGET_COL = "safety_index"

knn_feature_cols = [
    "dist_nearest_labeled_city",
    "log1p_dist_nearest_labeled_city",
    "crime_nearest_labeled_city",
    "safety_nearest_labeled_city",
    "same_country_as_nearest_labeled",
    "avg_crime_k5",
    "avg_safety_k5",
    "avg_crime_k10",
    "avg_safety_k10",
    "wavg_crime_k5",
    "wavg_safety_k5",
    "log1p_num_labeled_within_50km",
    "log1p_num_labeled_within_100km",
    "log1p_num_labeled_within_250km",
    "avg_crime_same_country_k5",
    "avg_safety_same_country_k5",
    "log1p_num_same_country_within_250km",
]

density_gravity_feature_cols = [
    "log1p_num_cities_50km",
    "log1p_sum_pop_50km",
    "log1p_pop_gravity_50km",
    "log1p_num_cities_100km",
    "log1p_sum_pop_100km",
    "log1p_pop_gravity_100km",
    "dist_to_nearest_large_city",
    "log1p_dist_to_nearest_large_city",
    "log1p_pop_of_nearest_large_city",
]

base_feature_cols = [
    "lat",
    "lon",
    "crimeindex_2020",
    "crimeindex_2023",
    "safetyindex_2020",
    "age_0_14",
    "age_15_64",
    "age_65_plus",
    "population",
    "density_per_km2",
]
# New v5 macro feature group

macro_cols_v5 = [
    "gdp",
    "gdp_per_capita",
    "unemployment",
    "homicide_rate",
    "life_expectancy_male",
    "life_expectancy_female",
    "infant_mortality",
    "urban_population_growth",
    "tourists",
]

# Build final v5 feature list = all v4 + macro

feature_cols_v5 = (
    knn_feature_cols
    + density_gravity_feature_cols
    + base_feature_cols
    + macro_cols_v5
)
# Remove duplicates while preserving order
feature_cols_v5 = list(dict.fromkeys(feature_cols_v5))

# Keep only columns that actually exist in df
missing_cols = [c for c in feature_cols_v5 if c not in df.columns]
if missing_cols:
    print("\nMissing columns not found in df:")
    print(missing_cols)

feature_cols_v5 = [c for c in feature_cols_v5 if c in df.columns]

print("\nNumber of v5 features actually used:", len(feature_cols_v5))
print(feature_cols_v5)


Number of v5 features actually used: 45
['dist_nearest_labeled_city', 'log1p_dist_nearest_labeled_city', 'crime_nearest_labeled_city', 'safety_nearest_labeled_city', 'same_country_as_nearest_labeled', 'avg_crime_k5', 'avg_safety_k5', 'avg_crime_k10', 'avg_safety_k10', 'wavg_crime_k5', 'wavg_safety_k5', 'log1p_num_labeled_within_50km', 'log1p_num_labeled_within_100km', 'log1p_num_labeled_within_250km', 'avg_crime_same_country_k5', 'avg_safety_same_country_k5', 'log1p_num_same_country_within_250km', 'log1p_num_cities_50km', 'log1p_sum_pop_50km', 'log1p_pop_gravity_50km', 'log1p_num_cities_100km', 'log1p_sum_pop_100km', 'log1p_pop_gravity_100km', 'dist_to_nearest_large_city', 'log1p_dist_to_nearest_large_city', 'log1p_pop_of_nearest_large_city', 'lat', 'lon', 'crimeindex_2020', 'crimeindex_2023', 'safetyindex_2020', 'age_0_14', 'age_15_64', 'age_65_plus', 'population', 'density_per_km2', 'gdp', 'gdp_per_capita', 'unemployment', 'homicide_rate', 'life_expectancy_male', 'life_expectancy_fe

In [3]:
# Final NA checks

print("\nNaNs in v5 features before fill:")
na_counts = df[feature_cols_v5].isna().sum()
print(na_counts[na_counts > 0])

# Median fill as a final safety net
for col in feature_cols_v5:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

# Drop rows with missing target only
df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)

print("\nRemaining NaNs in v5 features:", df[feature_cols_v5].isna().sum().sum())
print("Shape after target cleanup:", df.shape)


NaNs in v5 features before fill:
Series([], dtype: int64)

Remaining NaNs in v5 features: 0
Shape after target cleanup: (509, 77)


In [4]:
# Train/test split

X = df[feature_cols_v5].copy()
y = df[TARGET_COL].copy()

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index,
    test_size=0.2,
    random_state=42
)

print("\nTrain shape:", X_train.shape)
print("Test shape :", X_test.shape)


Train shape: (407, 45)
Test shape : (102, 45)


In [5]:
# Scale for MLP

scaler_v5 = StandardScaler()
X_train_scaled = scaler_v5.fit_transform(X_train)
X_test_scaled = scaler_v5.transform(X_test)

In [6]:
# Train MLP v5

mlp_v5 = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    alpha=0.0005,
    learning_rate_init=0.001,
    max_iter=3000,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=50,
    random_state=42
)

mlp_v5.fit(X_train_scaled, y_train)

MLPRegressor(alpha=0.0005, early_stopping=True,
             hidden_layer_sizes=(128, 64, 32), max_iter=3000,
             n_iter_no_change=50, random_state=42)

In [10]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
# Evaluate MLP v5

y_pred_train_mlp = mlp_v5.predict(X_train_scaled)
y_pred_test_mlp = mlp_v5.predict(X_test_scaled)

# MLP metrics
mlp_train_mae = mean_absolute_error(y_train, y_pred_train_mlp)
mlp_test_mae = mean_absolute_error(y_test, y_pred_test_mlp)

mlp_train_mse = mean_squared_error(y_train, y_pred_train_mlp)
mlp_test_mse = mean_squared_error(y_test, y_pred_test_mlp)

mlp_train_rmse = np.sqrt(mlp_train_mse)
mlp_test_rmse = np.sqrt(mlp_test_mse)

mlp_train_r2 = r2_score(y_train, y_pred_train_mlp)
mlp_test_r2 = r2_score(y_test, y_pred_test_mlp)
print("\n================ MLP v5 Results ================")
print("Train MAE :", mlp_train_mae)
print("Test MAE  :", mlp_test_mae)
print("Train RMSE:", mlp_train_rmse)
print("Test RMSE :", mlp_test_rmse)
print("Train R2  :", mlp_train_r2)
print("Test R2   :", mlp_test_r2)
print("Iterations:", mlp_v5.n_iter_)


================ MLP v5 Results ================
Train MAE : 5.171891240280824
Test MAE  : 8.317819425551365
Train RMSE: 6.835211686500969
Test RMSE : 10.670626950469076
Train R2  : 0.798022855904956
Test R2   : 0.5323152805183942
Iterations: 234


# Random forest to compare

In [11]:
# Train Random Forest v5 challenger

rf_v5 = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_v5.fit(X_train, y_train)

RandomForestRegressor(min_samples_leaf=2, min_samples_split=4, n_estimators=500,
                      n_jobs=-1, random_state=42)

In [13]:
# Evaluate Random Forest v5

# Predictions
y_pred_train_rf = rf_v5.predict(X_train)
y_pred_test_rf = rf_v5.predict(X_test)

# MAE
rf_train_mae = mean_absolute_error(y_train, y_pred_train_rf)
rf_test_mae = mean_absolute_error(y_test, y_pred_test_rf)

# MSE then RMSE (no squared kwarg in your sklearn)
rf_train_mse = mean_squared_error(y_train, y_pred_train_rf)
rf_test_mse = mean_squared_error(y_test, y_pred_test_rf)

rf_train_rmse = np.sqrt(rf_train_mse)
rf_test_rmse = np.sqrt(rf_test_mse)

# R²
rf_train_r2 = r2_score(y_train, y_pred_train_rf)
rf_test_r2 = r2_score(y_test, y_pred_test_rf)

print("\n============= Random Forest v5 Results =============")
print("Train MAE :", rf_train_mae)
print("Test MAE  :", rf_test_mae)
print("Train RMSE:", rf_train_rmse)
print("Test RMSE :", rf_test_rmse)
print("Train R2  :", rf_train_r2)
print("Test R2   :", rf_test_r2)


============= Random Forest v5 Results =============
Train MAE : 2.541540774558502
Test MAE  : 6.5111861948052
Train RMSE: 3.6646686088554996
Test RMSE : 8.926443367288705
Train R2  : 0.9419412468516191
Test R2   : 0.6727119003351527


In [14]:
# Build test prediction table

results_v5 = df.loc[idx_test, ["city", "country", TARGET_COL]].copy()

results_v5["mlp_pred_v5"] = y_pred_test_mlp
results_v5["mlp_abs_error_v5"] = (results_v5[TARGET_COL] - results_v5["mlp_pred_v5"]).abs()

results_v5["rf_pred_v5"] = y_pred_test_rf
results_v5["rf_abs_error_v5"] = (results_v5[TARGET_COL] - results_v5["rf_pred_v5"]).abs()

results_v5 = results_v5.sort_values("mlp_abs_error_v5").reset_index(drop=True)

print("\nPrediction sample:")
print(results_v5.head(20).to_string(index=False))


Prediction sample:
              city              country  safety_index  mlp_pred_v5  mlp_abs_error_v5  rf_pred_v5  rf_abs_error_v5
       Phoenix, AZ        United States         47.77    47.793799          0.023799   47.728984         0.041016
           Sharjah United Arab Emirates         84.10    83.927332          0.172668   84.664158         0.564158
      Bloemfontein         South Africa         28.68    28.453197          0.226803   24.924483         3.755517
           Algiers              Algeria         47.40    47.092818          0.307182   47.454912         0.054912
            Denver                   CO         54.70    55.141308          0.441308   56.132392         1.432392
Salt Lake City, UT        United States         66.89    66.431787          0.458213   63.467144         3.422856
      Honolulu, HI        United States         53.76    53.161996          0.598004   53.757448         0.002552
            Darwin            Australia         39.60    38.979377  

In [15]:
# Simple feature importance from RF

rf_importance_df = pd.DataFrame({
    "feature": feature_cols_v5,
    "importance": rf_v5.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

print("\nTop RF feature importances:")
print(rf_importance_df.head(20).to_string(index=False))


Top RF feature importances:
                         feature  importance
                 crimeindex_2023    0.436276
                   wavg_crime_k5    0.121795
                  wavg_safety_k5    0.098669
                safetyindex_2020    0.033203
                 crimeindex_2020    0.024985
              log1p_sum_pop_50km    0.021235
             log1p_sum_pop_100km    0.018596
 log1p_pop_of_nearest_large_city    0.017505
      crime_nearest_labeled_city    0.013817
     safety_nearest_labeled_city    0.013654
                             lon    0.012923
      avg_safety_same_country_k5    0.012906
 log1p_dist_nearest_labeled_city    0.011124
       dist_nearest_labeled_city    0.010986
       avg_crime_same_country_k5    0.010431
                             lat    0.010238
                   avg_crime_k10    0.010085
           log1p_num_cities_50km    0.009442
                  avg_safety_k10    0.008516
log1p_dist_to_nearest_large_city    0.007760


In [16]:
# Save artifacts

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

output_dir = Path("../data/model_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(mlp_v5, models_dir / "mlp_v5.pkl")
joblib.dump(scaler_v5, models_dir / "scaler_v5.pkl")
joblib.dump(rf_v5, models_dir / "rf_v5.pkl")

results_v5.to_csv(output_dir / "v5_test_predictions.csv", index=False)
rf_importance_df.to_csv(output_dir / "v5_rf_feature_importance.csv", index=False)

with open(output_dir / "v5_feature_columns.txt", "w", encoding="utf-8") as f:
    for col in feature_cols_v5:
        f.write(col + "\n")

metrics_v5 = pd.DataFrame([
    {
        "model": "MLPRegressor_v5",
        "train_mae": mlp_train_mae,
        "test_mae": mlp_test_mae,
        "train_rmse": mlp_train_rmse,
        "test_rmse": mlp_test_rmse,
        "train_r2": mlp_train_r2,
        "test_r2": mlp_test_r2,
        "n_features": len(feature_cols_v5),
        "n_train": len(X_train),
        "n_test": len(X_test),
    },
    {
        "model": "RandomForestRegressor_v5",
        "train_mae": rf_train_mae,
        "test_mae": rf_test_mae,
        "train_rmse": rf_train_rmse,
        "test_rmse": rf_test_rmse,
        "train_r2": rf_train_r2,
        "test_r2": rf_test_r2,
        "n_features": len(feature_cols_v5),
        "n_train": len(X_train),
        "n_test": len(X_test),
    }
])

metrics_v5.to_csv(output_dir / "v5_model_metrics.csv", index=False)

print("\nSaved artifacts:")
print(models_dir / "mlp_v5.pkl")
print(models_dir / "scaler_v5.pkl")
print(models_dir / "rf_v5.pkl")
print(output_dir / "v5_test_predictions.csv")
print(output_dir / "v5_rf_feature_importance.csv")
print(output_dir / "v5_feature_columns.txt")
print(output_dir / "v5_model_metrics.csv")


Saved artifacts:
..\models\mlp_v5.pkl
..\models\scaler_v5.pkl
..\models\rf_v5.pkl
..\data\model_outputs\v5_test_predictions.csv
..\data\model_outputs\v5_rf_feature_importance.csv
..\data\model_outputs\v5_feature_columns.txt
..\data\model_outputs\v5_model_metrics.csv


# Summary 

v5 adds macro-economic features on top of the v4 KNN+worldpop feature set, but a single fixed MLP is clearly worse than both the flexible v4 MLP search and the v5 Random Forest, which suggests (a) we should bring back the v4-style multi-variant MLP search, and (b) we may want to slim features a bit based on RF importance for a v6 

## Data and features:

509 labeled cities, 77 raw columns in the v5 CSV.

Final feature set used in v5: 45 numeric features = all v4 KNN + density/gravity + base geo/demographic features, plus 9 macro columns (gdp, gdp_per_capita, unemployment, homicide_rate, life_expectancy_male, life_expectancy_female, infant_mortality, urban_population_growth, tourists).

No remaining NaNs in the selected features after median fill; target safety_index has 509 non-null values.

## MLP v5 (single scikit-learn model, (128, 64, 32)):

Train: MAE ≈ 5.17, RMSE ≈ 6.84, R² ≈ 0.80.

Test: MAE ≈ 8.32, RMSE ≈ 10.67, R² ≈ 0.53.

This is only marginally better than the v1/v3 baselines and actually worse than your best v4 PyTorch MLP (RMSE ≈ 9.66, R² ≈ 0.62), which used a small grid over architectures and regularization.

## Random Forest v5:

Train: MAE ≈ 2.54, RMSE ≈ 3.66, R² ≈ 0.94 (heavy overfit, as expected).

Test: MAE ≈ 6.51, RMSE ≈ 8.93, R² ≈ 0.67 — meaningfully better than the v5 MLP and also better than the v4 best MLP.

This reinforces the usual pattern that tree ensembles are very strong on engineered tabular data, and your macros + KNN features are highly exploitable by RF.

## Comparison to v4:

v4 best model (safety_knn_v1_deeper) with KNN + worldpop but no macros: Test RMSE ≈ 9.66, MAE ≈ 7.32, R² ≈ 0.62.

v5 MLP with macros: Test RMSE ≈ 10.67, R² ≈ 0.53 → clearly worse than v4.

v5 RF with macros: Test RMSE ≈ 8.93, R² ≈ 0.67 → best overall so far.

# Conclusion:

the macro features are useful (RF improves), but the single scikit-learn MLP is under-tuned relative to v4 PyTorch search, so to fairly test “v5 MLP + macros” we want the same v4-style multi-variant training loop on the v5 feature set.

In [17]:
import ipynbname

nb_path = ipynbname.path()
nb_name = nb_path.name 
nb_name

import sys
from pathlib import Path
 
!"{sys.executable}" -m nbconvert --to pdf "{nb_name}"

[NbConvertApp] Converting notebook Geo_MLP_v5_knn.ipynb to pdf
[NbConvertApp] Writing 61540 bytes to notebook.tex
[NbConvertApp] Building PDF
[NbConvertApp] Running xelatex 3 times: ['xelatex', 'notebook.tex', '-quiet']
[NbConvertApp] Running bibtex 1 time: ['bibtex', 'notebook']
[NbConvertApp] WARNING | b had problems, most likely because there were no citations
[NbConvertApp] PDF successfully created
[NbConvertApp] Writing 56074 bytes to Geo_MLP_v5_knn.pdf
